# 02 Vectorization: TF-IDF and sentence embeddings

Compares the two vector representations of the reports:

| | TF-IDF | Sentence embeddings |
|---|---|---|
| Input | `text_strong` (lemmas, without stopwords) | `text_light` (word forms, sentence context) |
| Used for | NMF | KMeans |

Requires `python -m src.preprocessing.pipeline` and `python -m src.vectorization.pipeline` (computes the embedding cache).

In [ ]:
import os
import sys

# Make the project root importable (the notebook runs in notebooks/).
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from src.config import load_config
from src.vectorization.compare import describe, nearest_neighbors, neighbor_overlap
from src.vectorization.pipeline import embeddings_from_config, load_processed, tfidf_from_config

cfg = load_config()
reports = load_processed(cfg)
vectorizer, tfidf = tfidf_from_config(reports, cfg)
embeddings = embeddings_from_config(reports, cfg)
len(reports)

## D1 Key figures

In [ ]:
# One row per representation. Replacing a whole column may change its type.
figures = pd.DataFrame({"TF-IDF": describe(tfidf), "Embeddings": describe(embeddings)}).T
figures = figures.astype({"rows": int, "dimensions": int, "zero_rows": int})
figures["density"] = figures["density"].map("{:.4%}".format)
figures["memory_mb"] = figures["memory_mb"].round(1)
figures

## D2 TF-IDF vocabulary

Share of word pairs and numbers among the terms, and the terms with the lowest IDF (= occurring in the most reports).

In [ ]:
terms = pd.Series(vectorizer.get_feature_names_out())
idf = pd.Series(vectorizer.idf_, index=terms)
print("terms:", len(terms))
print("word pairs:", terms.str.contains(" ").sum())
print("terms with digits:", terms.str.contains(r"\d").sum())
idf.sort_values().head(20)

Reports without any TF-IDF term: empty after strong cleaning (48), or only terms below `min_df` or excluded words.
NMF cannot assign them to a topic (see block E).

In [ ]:
zero = tfidf.getnnz(axis=1) == 0
print("reports without TF-IDF term:", zero.sum(), f"({zero.mean():.2%})")
print("of these empty after strong cleaning:", (reports.loc[zero, "text_strong"] == "").sum())

## D3 Text length and the input limit of the embedding model

The model reads at most 128 tokens (word pieces). Longer texts are truncated.

In [ ]:
from transformers import AutoTokenizer

emb_cfg = cfg["embeddings"]
tokenizer = AutoTokenizer.from_pretrained(emb_cfg["model"], revision=emb_cfg["revision"])
n_tokens = pd.Series([len(ids) for ids in tokenizer(reports["text_light"].tolist())["input_ids"]])
print(n_tokens.describe().round(1))
print("truncated (> 128 tokens):", (n_tokens > 128).sum(), f"({(n_tokens > 128).mean():.2%})")

## D4 Do both representations see the same reports as similar?

For a random sample of reports, compare the 10 nearest neighbours in both representations.
Overlap 1 = same neighbours, 0 = no common neighbour. Reports without a TF-IDF term are excluded.

In [ ]:
K = 10
N_QUERIES = 500
rng = np.random.default_rng(cfg["seed"])
queries = rng.choice(np.flatnonzero(~zero), size=N_QUERIES, replace=False)
overlap = neighbor_overlap(tfidf, embeddings, queries, K)
print(f"mean neighbour overlap (k={K}, {N_QUERIES} reports): {overlap:.2f}")

Example: nearest neighbours of one report in both representations.
Outputs are not committed (nbstripout), because report texts may contain personal data.

In [ ]:
def show_neighbors(query, k=5):
    print("REPORT:", reports.loc[query, "text_light"], "\n")
    for name, matrix in [("TF-IDF", tfidf), ("Embeddings", embeddings)]:
        print(name)
        for i in nearest_neighbors(matrix, query, k):
            print("  -", reports.loc[i, "text_light"][:120])
        print()

show_neighbors(queries[0])

Synonyms: "Müll" and "Abfall" are merged by the synonym list (`preprocessing.synonyms` in the config),
"Beleuchtung" and "Lampe" are not. Compared are reports that consist only of one of these words.

In [ ]:
def similarity(a, b):
    """Similarity of the first reports with exactly these texts (0 = nothing in common, 1 = same)."""
    i = reports.index[reports["text_light"] == a][0]
    j = reports.index[reports["text_light"] == b][0]
    return (tfidf[i] @ tfidf[j].T).toarray()[0, 0], float(embeddings[i] @ embeddings[j])

for a, b in [("Müll", "Abfall"), ("Beleuchtung", "Lampe")]:
    in_tfidf, in_embeddings = similarity(a, b)
    print(f"{a} / {b}: TF-IDF {in_tfidf:.2f}, embeddings {in_embeddings:.2f}")

query = reports.index[reports["text_light"] == "Müll liegt rum"][0]
show_neighbors(query, k=10)

## Interpretation

Structure: TF-IDF is sparse (8,761 terms, 0.08 % non-zero) and every column is a word, so NMF topics can be read
as word lists. State words such as "nicht" or "seit", meta words such as "bitte" and pure numbers are not used as
terms (`tfidf.exclude_words`). The embeddings are dense (384 values) and their dimensions have no meaning of their
own. For KMeans, topic words have to come from the texts in each cluster.

Similarity: TF-IDF only finds reports that share lemmas. Synonyms only match if the synonym list merges them. On a
scale from 0 (nothing in common) to 1 (same content), "Müll" and "Abfall" score 1 in TF-IDF, "Beleuchtung" and
"Lampe" score 0. The embeddings score both pairs at 0.85 and 0.88 without any list. Only 13 % of the 10 nearest
neighbours are the same in both representations, so NMF and KMeans will probably find partly different topics.

Short texts: 1,508 reports (1.9 %) have no TF-IDF vector. They are empty after strong cleaning (48), or contain only
rare terms or only excluded words (e.g. "Seit Wochen"). NMF cannot assign them, so they are reported separately in
block E. Embeddings cover every report, but for very short texts with a name (e.g. a stop) the neighbours can be
unrelated. Truncation at 128 tokens affects only 3.1 % of the texts.

The agreement is based on 500 random reports, the qualitative observations on a few examples.